In [1]:
pip install numpy scikit-learn sentence-transformers pandas

In [2]:
import numpy, sklearn, sentence_transformers, pandas
print("numpy", numpy.__version__)
print("scikit-learn", sklearn.__version__)
print("sentence-transformers", sentence_transformers.__version__)
print("pandas", pandas.__version__)

numpy 2.4.6
scikit-learn 1.6.1
sentence-transformers 5.7.0
pandas 2.2.3


In [3]:
%%writefile lab6_search.py
"""Retrieval helper used by every script: TF-IDF or sentence-embedding search."""
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

ENCODER = "sentence-transformers/all-MiniLM-L6-v2"


class Retriever:
    """Index a list of records ({"id": ..., "text": ...}) and return the best matches for a query."""

    def __init__(self, records, backend="tfidf"):
        if not records or any(not rec["text"].strip() for rec in records):
            raise ValueError("Provide nonempty records")
        if backend not in ("tfidf", "embedding"):
            raise ValueError("Use tfidf or embedding")
        self.records = records
        self.backend = backend
        passages = [rec["text"] for rec in records]

        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(passages)
        else:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(ENCODER, device="cpu")
            self.check_length(passages)
            self.matrix = self.model.encode(passages, normalize_embeddings=True)

    def check_length(self, texts):
        """Reject any text longer than the encoder limit (special tokens included)."""
        limit = self.model.max_seq_length
        for text in texts:
            ids = self.model.tokenizer(text, truncation=False)["input_ids"]
            if len(ids) > limit:
                raise ValueError("Input exceeds encoder token limit")

    def _score_all(self, query):
        """Return one score per record, or None when TF-IDF finds no shared word."""
        if self.backend == "tfidf":
            vec = self.vectorizer.transform([query])
            if vec.nnz == 0:
                return None
            return (vec @ self.matrix.T).toarray()[0]
        self.check_length([query])
        vec = self.model.encode([query], normalize_embeddings=True)[0]
        return self.matrix @ vec

    def search(self, query, k=3, filters=None):
        """Top-k (record, score) pairs, highest score first; metadata filters apply first."""
        if not query.strip() or k <= 0:
            return []
        scores = self._score_all(query)
        if scores is None:
            return []
        wanted = filters or {}
        allowed = [i for i, rec in enumerate(self.records)
                   if all(rec.get(key) == value for key, value in wanted.items())]
        allowed.sort(key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in allowed[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])

Writing lab6_search.py


In [4]:
%%writefile task1_tokens.py
"""Task 1: compare whitespace, word-like (regex) and model sub-word tokens."""
import re

samples = [
    "AI, search!",              # punctuation
    "I can't log in.",          # contraction
    "help@example.com",         # email
    "یہ ایک مثال ہے",            # Urdu
    "",                         # empty string
    "CS4944 version 2.1",       # identifier and number
]


def whitespace_tokens(text):
    return text.split()


def word_like_tokens(text):
    return re.findall(r"\b\w+\b", text, flags=re.UNICODE)


print("=== Whitespace vs word-like tokens ===")
for text in samples:
    plain = whitespace_tokens(text)
    wordy = word_like_tokens(text)
    print(f"Original : {text!r}")
    print(f"  whitespace -> {plain}  ({len(plain)} tokens)")
    print(f"  word-like  -> {wordy}  ({len(wordy)} tokens)")
assert word_like_tokens("") == [] and whitespace_tokens("") == []

print()
print("=== Normalization risk ===")
risky = "I can't log in."
stripped = re.sub(r"[^\w\s]", "", risky.lower())
print(f"Original   : {risky!r}")
print(f"Normalized : {stripped!r}  <- the apostrophe is gone, so 'can't' became 'cant'")

print()
print("=== Model sub-word tokens (all-MiniLM-L6-v2) ===")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
for text in samples[:2]:
    print(f"{text!r} -> {model.tokenizer.tokenize(text)}")

Writing task1_tokens.py


In [5]:
!python task1_tokens.py

=== Whitespace vs word-like tokens ===
Original : 'AI, search!'
  whitespace -> ['AI,', 'search!']  (2 tokens)
  word-like  -> ['AI', 'search']  (2 tokens)
Original : "I can't log in."
  whitespace -> ['I', "can't", 'log', 'in.']  (4 tokens)
  word-like  -> ['I', 'can', 't', 'log', 'in']  (5 tokens)
Original : 'help@example.com'
  whitespace -> ['help@example.com']  (1 tokens)
  word-like  -> ['help', 'example', 'com']  (3 tokens)
Original : 'یہ ایک مثال ہے'
  whitespace -> ['یہ', 'ایک', 'مثال', 'ہے']  (4 tokens)
  word-like  -> ['یہ', 'ایک', 'مثال', 'ہے']  (4 tokens)
Original : ''
  whitespace -> []  (0 tokens)
  word-like  -> []  (0 tokens)
Original : 'CS4944 version 2.1'
  whitespace -> ['CS4944', 'version', '2.1']  (3 tokens)
  word-like  -> ['CS4944', 'version', '2', '1']  (4 tokens)

=== Normalization risk ===
Original   : "I can't log in."
Normalized : 'i cant log in'  <- the apostrophe is gone, so 'can't' became 'cant'

=== Model sub-word tokens (all-MiniLM-L6-v2) ===
modules.j

In [6]:
%%writefile guided_chatbot.py
"""Command-line FAQ chatbot: exact questions get a stored answer, others get suggestions."""
from lab6_search import Retriever

FAQ = [
    {"id": "faq:password", "question": "How do I reset my password?",
     "answer": "Use the recovery link on the login page.",
     "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?",
     "answer": "Use the assignment upload page in the learning portal.",
     "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?",
     "answer": "Request an extension with medical evidence before the deadline.",
     "source": "handbook.txt"},
    # Two FAQs added for Task 1 (synthetic teaching data)
    {"id": "faq:fees", "question": "When is the semester fee due?",
     "answer": "Pay the semester fee at the accounts office before the first day of classes.",
     "source": "fee_rules.txt"},
    {"id": "faq:library", "question": "How many books can I borrow from the library?",
     "answer": "Show your student card at the library desk to borrow up to five books.",
     "source": "library_guide.txt"},
]


def build_bot(backend="tfidf"):
    records = [dict(item, text=item["question"] + " " + item["answer"]) for item in FAQ]
    return Retriever(records, backend)


def reply(engine, query):
    if not query.strip():
        return "Please enter a question."

    # Only an exact canonical question is answered automatically
    asked = query.strip().casefold()
    for item in FAQ:
        if asked == item["question"].casefold():
            return f"{item['answer']} [source: {item['source']}]"

    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."

    lines = ["Possible matches; choose or rephrase your question:"]
    lines += [f"{rec['question']} (similarity {score:.3f})" for rec, score in hits]
    return "\n".join(lines)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--backend", choices=["tfidf", "embedding"], default="tfidf")
    bot = build_bot(parser.parse_args().backend)

    print("FAQ assistant. Type exit to stop.")
    while True:
        try:
            text = input("You: ")
        except (EOFError, KeyboardInterrupt):
            break
        if text.strip().lower() == "exit":
            break
        print("Bot:", reply(bot, text))

Writing guided_chatbot.py


In [7]:
from guided_chatbot import build_bot, reply

bot = build_bot("tfidf")
conversation = [
    "How do I reset my password?",                      # exact FAQ question
    "password recovery",                                # paraphrase -> suggestions
    "How many books can I borrow from the library?",    # exact (new FAQ)
    "when is fee payment due",                          # paraphrase of a new FAQ
    "",                                                 # empty input
    "volcano geology",                                  # unsupported question
]
for question in conversation:
    print("You:", question)
    print("Bot:", reply(bot, question))
    print()

You: How do I reset my password?
Bot: Use the recovery link on the login page. [source: it_help.txt]

You: password recovery
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.444)
Where do I submit my assignment? (similarity 0.000)
Can illness justify a late submission? (similarity 0.000)

You: How many books can I borrow from the library?
Bot: Show your student card at the library desk to borrow up to five books. [source: library_guide.txt]

You: when is fee payment due
Bot: Possible matches; choose or rephrase your question:
When is the semester fee due? (similarity 0.522)
How do I reset my password? (similarity 0.000)
Where do I submit my assignment? (similarity 0.000)

You: 
Bot: Please enter a question.

You: volcano geology
Bot: No matching FAQ. Please contact the help desk.



In [8]:
from guided_chatbot import build_bot, reply

bot = build_bot("tfidf")
conversation = [
    "How do I reset my password?",                      # exact FAQ question
    "password recovery",                                # paraphrase -> suggestions
    "How many books can I borrow from the library?",    # exact (new FAQ)
    "when is fee payment due",                          # paraphrase of a new FAQ
    "",                                                 # empty input
    "volcano geology",                                  # unsupported question
]
for question in conversation:
    print("You:", question)
    print("Bot:", reply(bot, question))
    print()

You: How do I reset my password?
Bot: Use the recovery link on the login page. [source: it_help.txt]

You: password recovery
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.444)
Where do I submit my assignment? (similarity 0.000)
Can illness justify a late submission? (similarity 0.000)

You: How many books can I borrow from the library?
Bot: Show your student card at the library desk to borrow up to five books. [source: library_guide.txt]

You: when is fee payment due
Bot: Possible matches; choose or rephrase your question:
When is the semester fee due? (similarity 0.522)
How do I reset my password? (similarity 0.000)
Where do I submit my assignment? (similarity 0.000)

You: 
Bot: Please enter a question.

You: volcano geology
Bot: No matching FAQ. Please contact the help desk.



In [9]:
assert "it_help.txt" in reply(bot, "How do I reset my password?")
assert reply(bot, "") == "Please enter a question."
assert "No matching FAQ" in reply(bot, "volcano geology")
assert "Possible matches" in reply(bot, "password recovery")
print("FAQ behaviour checks passed")

FAQ behaviour checks passed


In [10]:
%run guided_chatbot.py

FAQ assistant. Type exit to stop.
You: enter
Bot: No matching FAQ. Please contact the help desk.
You: How do I reset my password
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.624)
Where do I submit my assignment? (similarity 0.178)
How many books can I borrow from the library? (similarity 0.062)
You: exit


In [11]:
!pip install -q numpy scikit-learn sentence-transformers pandas

In [12]:
from google.colab import files

for name in ["lab6_search.py", "task1_tokens.py", "guided_chatbot.py"]:
    files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
!pip install -q numpy scikit-learn sentence-transformers pandas

In [14]:
%%writefile lab6_search.py
"""Retrieval helper used by every script: TF-IDF or sentence-embedding search."""
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

ENCODER = "sentence-transformers/all-MiniLM-L6-v2"


class Retriever:
    """Index a list of records ({"id": ..., "text": ...}) and return the best matches for a query."""

    def __init__(self, records, backend="tfidf"):
        if not records or any(not rec["text"].strip() for rec in records):
            raise ValueError("Provide nonempty records")
        if backend not in ("tfidf", "embedding"):
            raise ValueError("Use tfidf or embedding")
        self.records = records
        self.backend = backend
        passages = [rec["text"] for rec in records]

        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(passages)
        else:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(ENCODER, device="cpu")
            self.check_length(passages)
            self.matrix = self.model.encode(passages, normalize_embeddings=True)

    def check_length(self, texts):
        """Reject any text longer than the encoder limit (special tokens included)."""
        limit = self.model.max_seq_length
        for text in texts:
            ids = self.model.tokenizer(text, truncation=False)["input_ids"]
            if len(ids) > limit:
                raise ValueError("Input exceeds encoder token limit")

    def _score_all(self, query):
        """Return one score per record, or None when TF-IDF finds no shared word."""
        if self.backend == "tfidf":
            vec = self.vectorizer.transform([query])
            if vec.nnz == 0:
                return None
            return (vec @ self.matrix.T).toarray()[0]
        self.check_length([query])
        vec = self.model.encode([query], normalize_embeddings=True)[0]
        return self.matrix @ vec

    def search(self, query, k=3, filters=None):
        """Top-k (record, score) pairs, highest score first; metadata filters apply first."""
        if not query.strip() or k <= 0:
            return []
        scores = self._score_all(query)
        if scores is None:
            return []
        wanted = filters or {}
        allowed = [i for i, rec in enumerate(self.records)
                   if all(rec.get(key) == value for key, value in wanted.items())]
        allowed.sort(key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in allowed[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])

Overwriting lab6_search.py


In [15]:
%%writefile task2_chunks.py
"""Task 2: word-window and sentence-aware chunking with source offsets."""
import re


def word_spans(text):
    """(start, end) character span of every whitespace-separated word."""
    return [m.span() for m in re.finditer(r"\S+", text)]


def windows(text, size=40, overlap=10, offset=0):
    """Fixed word windows of `size` words; neighbours share `overlap` words."""
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    step = size - overlap
    result = []
    first = 0
    while first < len(spans):
        last = min(first + size, len(spans)) - 1
        result.append((offset + spans[first][0], offset + spans[last][1]))
        if last == len(spans) - 1:
            break
        first += step
    return result


def split_sentences(text):
    """Teaching splitter: cut after . ! or ? followed by whitespace."""
    edges = [0] + [m.end() for m in re.finditer(r"[.!?]\s+", text)] + [len(text)]
    return list(zip(edges, edges[1:]))


def sentence_chunks(text, size=40):
    """Pack whole sentences into chunks of at most `size` words.

    A sentence longer than `size` falls back to non-overlapping word windows.
    """
    if size <= 0:
        raise ValueError("size must be positive")
    chunks = []
    current = None                      # [start, end, word_count] of the chunk being built

    def close_current():
        nonlocal current
        if current is not None:
            chunks.append((current[0], current[1]))
            current = None

    for start, end in split_sentences(text):
        words = len(text[start:end].split())
        if words == 0:
            continue
        if words > size:
            close_current()
            chunks.extend(windows(text[start:end], size, 0, start))
        elif current is not None and current[2] + words <= size:
            current[1] = end
            current[2] += words
        else:
            close_current()
            current = [start, end, words]
    close_current()
    return chunks


def make_chunks(docs, strategy):
    """Chunk every document. strategy is "0", "10" (window overlap) or "sentence"."""
    result = []
    for doc_id, text in docs.items():
        if strategy == "sentence":
            spans = sentence_chunks(text)
        else:
            spans = windows(text, 40, int(strategy))
        for number, (start, end) in enumerate(spans):
            result.append({"id": f"{doc_id}:{number}", "doc": doc_id,
                           "start": start, "end": end, "text": text[start:end]})
    return result


if __name__ == "__main__":
    assert windows("") == []
    assert windows("short text") == [(0, 10)]
    try:
        windows("a b", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap was accepted")

    long_sentence = " ".join(["word"] * 95) + "."
    pieces = sentence_chunks(long_sentence)
    assert all(len(long_sentence[a:b].split()) <= 40 for a, b in pieces)
    print("95-word sentence ->", len(pieces), "fallback windows")

    sample = {"example": "Reset your password. Keep recovery codes private."}
    for strategy in ["0", "10", "sentence"]:
        for chunk in make_chunks(sample, strategy):
            assert chunk["text"] == sample[chunk["doc"]][chunk["start"]:chunk["end"]]
            print(strategy, chunk)
    print("All chunking checks passed")

Writing task2_chunks.py


In [16]:
!python task2_chunks.py

95-word sentence -> 3 fallback windows
0 {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
10 {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
sentence {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
All chunking checks passed


In [17]:
%%writefile create_dataset.py
"""Create corpus.json (5 synthetic documents, 150-250 words each) and queries.json (10 labeled queries).

All text is synthetic teaching data written for this lab, not real university policy.
Evidence strings are exact substrings of the documents and include the rule and its exception.
"""
import json

docs = {
"late": "Assessment Policy for Undergraduate Courses. All assignments must be submitted through the learning portal before the stated deadline, and the portal records the exact time of every upload. Late submission is normally penalized by ten percent of the total mark for every day that passes after the deadline, and work that arrives more than five days late receives no marks at all. Students who cannot meet a deadline because of illness may request an extension, but the request is accepted only if the student sends a medical certificate to the course coordinator before the deadline passes, and the certificate must be signed by a registered doctor and must clearly state the dates on which the student was unfit to study. Technical problems with the portal are handled separately: a student who experiences a portal failure should take a screenshot of the error message and email it to the help desk within two hours. The coordinator may also grant short extensions for family emergencies when written proof is provided. Group projects follow the same rules, but every member must confirm the request. Students should read this policy carefully at the start of the semester, because the deadlines listed in each course outline are final and are not reopened for discussion once marking has started.",
"password": "IT Help Desk: Account and Password Guide. Every student receives a university account on the first day of enrolment, and the account gives access to email, the learning portal and the library catalogue. A forgotten password can be reset using the recovery link on the login page, and the link is sent only to the personal email address that the student registered during enrolment. The recovery link expires after 24 hours, so students who do not use it in time must request a new one, and after three failed attempts the account is locked for thirty minutes to protect it from strangers. Staff members at the help desk will never ask a student to share a password or a recovery code over the phone or by email, and any message that asks for such details should be reported at once. Older versions of this guide stated that recovery links expire after 12 hours, but that rule applied only to accounts created before the 2020 system upgrade and is no longer in use. Students who no longer have access to their registered email address must visit the help desk in person with their student card, where a staff member can verify their identity and update the contact details.",
"assignment": "Assignment Submission Guide. Assignments are uploaded through the learning portal, and each course page contains a submission box that opens automatically when the assignment is released. Upload assignments as a single PDF file, because the portal rejects documents in other formats, and name the file with the student number followed by the course code so that markers can find it easily. The maximum file size is twenty megabytes, and larger files should be compressed or split into appendices that are uploaded as separate attachments, except for source code, which must always be submitted as one archive. Students may replace a submitted file as many times as they like before the deadline, and only the most recent upload is marked. After the deadline the submission box closes, and a closed box can be reopened only by the course coordinator. A confirmation email is sent after every successful upload; students who do not receive it within ten minutes should check the portal history before contacting the help desk. Handwritten work is accepted only when the course outline says so, and in that case a clear scan must be uploaded instead of the paper copy. Plagiarism checks run automatically on every upload, and similarity reports are visible only to the marker.",
"contractor": "Remote Work Policy. Employees may request remote work for up to two days each week with the written approval of their line manager, and the arrangement is reviewed every six months to make sure that it still suits both the employee and the team. Contractors can work remotely only if the signed contract explicitly permits it, and a contractor whose contract is silent on the subject must come to the office until the contract is formally amended by the human resources department. Employee remote work rules do not apply to contractors, and managers are not allowed to promise remote work to a contractor verbally. All remote workers must use the company laptop and the secure virtual private network, and company documents must never be stored on personal devices or shared through personal email accounts. During remote days, staff are expected to be reachable between ten in the morning and four in the afternoon, and meetings should be attended with the camera on unless the organiser says otherwise. Some roles, such as reception, laboratory support and equipment maintenance, require physical presence and are therefore excluded from remote work altogether, regardless of seniority. Employees who wish to work from another city for more than a week must apply separately.",
"printer": "Printer Troubleshooting Guide. This guide covers the shared network printers found in the library and the computer laboratories. For blank printer pages, check the ink levels first and then run the cleaning utility from the printer menu, and if the pages are still blank after the cleaning cycle, print a test page to confirm whether the problem is in the printer or in the document. Never open the printer while it is switched on, and always wait ten minutes after printing before touching the fuser unit, because it stays hot enough to cause burns. Paper jams should be cleared by pulling the paper gently in the direction of the paper path, and torn pieces must be removed completely before printing restarts. Low print quality is usually caused by dirty print heads or by the wrong paper type, so check the paper setting in the print dialog before replacing any cartridge. Scanners attached to the same machines are a separate device: for a blank scanner image, check document placement and select the correct scanning source instead. Faults that remain after these steps should be reported to the help desk with the printer label number and a photograph of the error message. Cartridges are replaced only by IT staff.",
}

ev = {
"late1": "Students who cannot meet a deadline because of illness may request an extension, but the request is accepted only if the student sends a medical certificate to the course coordinator before the deadline passes",
"late2": "the certificate must be signed by a registered doctor and must clearly state the dates on which the student was unfit to study",
"pw1": "A forgotten password can be reset using the recovery link on the login page, and the link is sent only to the personal email address that the student registered during enrolment",
"pw2": "The recovery link expires after 24 hours, so students who do not use it in time must request a new one",
"as1": "The maximum file size is twenty megabytes, and larger files should be compressed or split into appendices that are uploaded as separate attachments, except for source code, which must always be submitted as one archive",
"as2": "Handwritten work is accepted only when the course outline says so, and in that case a clear scan must be uploaded instead of the paper copy",
"co1": "Contractors can work remotely only if the signed contract explicitly permits it, and a contractor whose contract is silent on the subject must come to the office until the contract is formally amended by the human resources department",
"pr1": "For blank printer pages, check the ink levels first and then run the cleaning utility from the printer menu, and if the pages are still blank after the cleaning cycle, print a test page to confirm whether the problem is in the printer or in the document",
}

rows = [
("What is required for an illness extension?", "late", ev["late1"], "dev"),
("How do I reset a forgotten password?", "password", ev["pw1"], "dev"),
("What is the maximum upload size for assignments?", "assignment", ev["as1"], "dev"),
("Can contractors work remotely?", "contractor", ev["co1"], "dev"),
("What should I do when the printer prints blank pages?", "printer", ev["pr1"], "test"),
("How long is the password recovery link valid?", "password", ev["pw2"], "test"),
("Can I submit handwritten coursework?", "assignment", ev["as2"], "test"),
("What proof is needed when I was ill?", "late", ev["late2"], "test"),
("What are tomorrow's flight prices?", None, None, "test"),
("Who won the football match?", None, None, "test"),
]
queries = [dict(query=q, doc=d, evidence=e, split=s) for q, d, e, s in rows]

for q in queries:
    if q["doc"]:
        assert q["evidence"] in docs[q["doc"]], q["query"]
for k, t in docs.items():
    print(k, len(t.split()), "words")

json.dump(docs, open("corpus.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(queries, open("queries.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

Writing create_dataset.py


In [18]:
!python create_dataset.py

late 213 words
password 207 words
assignment 208 words
contractor 207 words
printer 207 words


In [19]:
from task2_chunks import windows, sentence_chunks, split_sentences

print("Empty text      :", windows(""), sentence_chunks(""))
print("Short document  :", windows("short text"), sentence_chunks("short text"))
try:
    windows("a b", 2, 2)
except ValueError as error:
    print("Invalid overlap : ValueError ->", error)

long_sentence = " ".join(["word"] * 95) + "."
spans = sentence_chunks(long_sentence)
print("Long sentence   :", len(spans), "chunks with word counts",
      [len(long_sentence[a:b].split()) for a, b in spans])

Empty text      : [] []
Short document  : [(0, 10)] [(0, 10)]
Invalid overlap : ValueError -> Require size > 0 and 0 <= overlap < size
Long sentence   : 3 chunks with word counts [40, 40, 15]


In [20]:
text = "Dr. Khan approved the extension. Submit the form by Friday."
for a, b in split_sentences(text):
    print(repr(text[a:b]))
print("-> 'Dr.' was treated as a full sentence because the splitter only looks for punctuation followed by a space.")

'Dr. '
'Khan approved the extension. '
'Submit the form by Friday.'
-> 'Dr.' was treated as a full sentence because the splitter only looks for punctuation followed by a space.


In [21]:
import json
import pandas as pd
from task2_chunks import make_chunks

docs = json.load(open("corpus.json", encoding="utf-8"))
queries = json.load(open("queries.json", encoding="utf-8"))
labeled = sorted({(q["doc"], q["evidence"]) for q in queries if q["doc"]})
original_words = sum(len(t.split()) for t in docs.values())
print("Documents:", len(docs), "| original words:", original_words, "| labeled evidence:", len(labeled))

setups = [("Word window", "0", "0"), ("Word window", "10", "10"), ("Sentence-aware", "sentence", "none")]
rows, survival = [], {}
for name, strategy, overlap in setups:
    chunks = make_chunks(docs, strategy)
    sizes = [len(c["text"].split()) for c in chunks]
    survives = [any(c["doc"] == d and e in c["text"] for c in chunks) for d, e in labeled]
    survival[f"{name} ({overlap})"] = survives
    rows.append({"Strategy": name, "Overlap": overlap, "Chunks": len(chunks),
                 "Avg words": round(sum(sizes) / len(sizes), 1),
                 "Repeated words": sum(sizes) - original_words,
                 "Evidence kept together": f"{sum(survives)} of {len(labeled)}"})
display(pd.DataFrame(rows))

detail = pd.DataFrame(survival, index=[f"{d}: {e[:45]}..." for d, e in labeled]).replace({True: "yes", False: "NO"})
display(detail)

Documents: 5 | original words: 1042 | labeled evidence: 8


,Strategy,Overlap,Chunks,Avg words,Repeated words,Evidence kept together
0,Word window,0,30,34.7,0,2 of 8
1,Word window,10,35,38.3,300,5 of 8
2,Sentence-aware,none,37,28.2,0,6 of 8


,Word window (0),Word window (10),Sentence-aware (none)
assignment: Handwritten work is accepted only when the co...,yes,NO,yes
"assignment: The maximum file size is twenty megabytes, an...",NO,yes,yes
contractor: Contractors can work remotely only if the sig...,NO,NO,yes
late: Students who cannot meet a deadline because o...,NO,yes,yes
late: the certificate must be signed by a registere...,yes,yes,NO
password: A forgotten password can be reset using the r...,NO,yes,yes
"password: The recovery link expires after 24 hours, so ...",NO,yes,yes
"printer: For blank printer pages, check the ink levels...",NO,NO,NO


In [22]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
passages = ["Reset a forgotten password using the recovery link.",
            "Submit assignments through the learning portal.",
            "Check ink levels when a printer produces blank pages."]
P = model.encode(passages, normalize_embeddings=True)
print("Embedding shape:", P.shape)
assert np.allclose(np.linalg.norm(P, axis=1), 1, atol=1e-5)

for query in ["How can I recover my login?", "I do not need a password reset"]:
    scores = (model.encode([query], normalize_embeddings=True) @ P.T)[0]
    print("\nQuery:", query)
    for i in np.argsort(-scores):
        print(f"  {i}  {scores[i]:.3f}  {passages[i]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (3, 384)

Query: How can I recover my login?
  0  0.627  Reset a forgotten password using the recovery link.
  1  0.072  Submit assignments through the learning portal.
  2  0.018  Check ink levels when a printer produces blank pages.

Query: I do not need a password reset
  0  0.713  Reset a forgotten password using the recovery link.
  2  0.151  Check ink levels when a printer produces blank pages.
  1  0.100  Submit assignments through the learning portal.


In [23]:
from lab6_search import Retriever
from task2_chunks import make_chunks

chunks = make_chunks(docs, "sentence")
for c in chunks:                                   # offsets must reproduce the source text exactly
    assert docs[c["doc"]][c["start"]:c["end"]] == c["text"]

engines = {"TF-IDF": Retriever(chunks, "tfidf"), "Embedding": Retriever(chunks, "embedding")}
paraphrases = [
    "How can I get back into my account if I forgot my login?",
    "What do I do when my printer gives empty paper?",
    "Am I allowed to hand in late work when I am sick?",
]
for query in paraphrases:
    print("\nQuery:", query)
    for name, engine in engines.items():
        hits = engine.search(query, k=1)
        if not hits:
            print(f"  {name:<9}: no matching records")
            continue
        record, score = hits[0]
        print(f"  {name:<9}: {record['id']} (score {score:.3f}, chars {record['start']}-{record['end']})")
        print("             evidence:", record["text"][:120], "...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Query: How can I get back into my account if I forgot my login?
  TF-IDF   : password:0 (score 0.252, chars 0-203)
             evidence: IT Help Desk: Account and Password Guide. Every student receives a university account on the first day of enrolment, and ...
  Embedding: password:1 (score 0.661, chars 203-382)
             evidence: A forgotten password can be reset using the recovery link on the login page, and the link is sent only to the personal e ...

Query: What do I do when my printer gives empty paper?
  TF-IDF   : printer:4 (score 0.184, chars 551-713)
             evidence: Paper jams should be cleared by pulling the paper gently in the direction of the paper path, and torn pieces must be rem ...
  Embedding: printer:1 (score 0.670, chars 129-348)
             evidence: For blank printer pages, check the ink levels first and then run the cleaning utility from the printer menu, and if the  ...

Query: Am I allowed to hand in late work when I am sick?
  TF-IDF   : contract

In [24]:
import os, shutil
from google.colab import files

os.makedirs("Task2", exist_ok=True)
!pip freeze > requirements.txt
for name in ["lab6_search.py", "task2_chunks.py", "create_dataset.py",
             "corpus.json", "queries.json", "requirements.txt"]:
    shutil.copy(name, "Task2")

print(os.listdir("Task2"))
shutil.make_archive("Task2", "zip", "Task2")
files.download("Task2.zip")

['task2_chunks.py', 'create_dataset.py', 'queries.json', 'lab6_search.py', 'requirements.txt', 'corpus.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>